## Analyze Morphospace for Sparse LMs, Dense Corresp, NSM using PCA, tSNE, and UMAP
---
Compares three representations of the same specimens side by side: the 28 sparse landmarks, the
dense (population) correspondences, and the NSM latent codes. All three come from the same
`all_vtk_files` order, so one `specimens` table (species / family / trait / color / marker, joined
from `lizard_species_list.csv`) drives every plot.

*Last edited 10 Sep 2026 by K. Wolcott*

In [ ]:
# Imports, paths, and config

import os, re, json, ast, torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import PCA
from PIL import Image, ImageDraw, ImageFont

from NSM.plotting import (load_mrk_json, pc_pair_grid, tsne_scores,
                          umap_scores, export_scores)
from NSM.morphometrics import gm_prcomp

# ---------------------------------------------------------------- TO DO: edit
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"         # atlas/builder run that produced the landmark sets
ATLAS_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")
CKPT         = "2500"                         # latent code checkpoint to analyze
# -----------------------------------------------------------------------------

cwd       = Path.cwd()
base_wd   = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

ATLAS_DIR       = ATLAS_ROOT / ATLAS_RUN / "atlas"
SPARSE_LM_DIR   = ATLAS_ROOT / ATLAS_RUN / "alignedLMs"
DENSE_LM_DIR    = ATLAS_ROOT / ATLAS_RUN / "population_correspondences"
SPARSE_MEAN_FN  = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
DENSE_MEAN_FN   = ATLAS_DIR / "atlas_dense_correspondences.mrk.json"

OUT_DIR = Path("pca_tsne_umap_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Load config and filenames
with open("model_params_config.json") as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from model_params_config.json\033[0m")

all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"{len(all_vtk_files)} meshes listed in config")

# Load NSM latent codes
CKPT_PATH = f"latent_codes/{CKPT}.pth"
latent_ckpt = torch.load(CKPT_PATH, map_location="cpu")
codes = latent_ckpt["latent_codes"]["weight"].detach().cpu().numpy()
print(f"Latent codes: {codes.shape}")
assert len(codes) == len(all_vtk_files), (
    f"{len(codes)} latent codes but {len(all_vtk_files)} meshes in config -- order/count mismatch")

### Specimen metadata (species / family / trait / color / marker)

In [ ]:
# Build specimen metadata table -- one row per mesh, in the same order as `codes`

SPECIES_CSV = "../lizard_species_list.csv"
pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'" + chr(34))
sdf["color"]  = sdf["color"].apply(ast.literal_eval)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")
print("Specimens dataframe head:\n", specimens.head())

# One color per broad clade (matches plot_family_color_legend elsewhere) 
family_base_colors = (specimens.drop_duplicates("broad_taxon_for_plotting")
                     .set_index("broad_taxon_for_plotting")["color"].to_dict())

### Load the three shape representations
Sparse landmarks and dense correspondences are both GPA-aligned/scaled already, so `gm_prcomp`
(Procrustes PCA, from `NSM.morphometrics`) runs directly on them. Latent codes get plain `sklearn`
PCA since they aren't Procrustes shape coordinates.

In [ ]:
# 28 sparse landmarks

sparse_coords = np.stack([load_mrk_json(SPARSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                          for f in all_vtk_files])
sparse_mean, _ = load_mrk_json(SPARSE_MEAN_FN)
print(f"Sparse landmarks: {sparse_coords.shape}")
assert sparse_mean.shape == sparse_coords.shape[1:], "sparse atlas / specimen landmark count mismatch"

pca_sparse = gm_prcomp(sparse_coords)
print(f"{pca_sparse['x'].shape[1]} non-trivial PCs from {sparse_coords.shape[1]*3} sparse coordinates")

In [ ]:
# Dense correspondences -- gm_prcomp here costs roughly 30s at ~5000 points x 3 dims x this many
# specimens; the full-SVD PCA runs once and every downstream tSNE/UMAP call reuses `pca_dense["x"]`.

dense_coords = np.stack([load_mrk_json(DENSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                         for f in all_vtk_files])
dense_mean, _ = load_mrk_json(DENSE_MEAN_FN)
print(f"Dense correspondences: {dense_coords.shape}")
assert dense_mean.shape == dense_coords.shape[1:], "dense atlas / specimen point count mismatch"

pca_dense = gm_prcomp(dense_coords)
print(f"{pca_dense['x'].shape[1]} non-trivial PCs from {dense_coords.shape[1]*3} dense coordinates")

In [ ]:
# NSM latent codes

pca_latent_model = PCA(n_components=5)
latent_scores = pca_latent_model.fit_transform(codes)
latent_prop = pca_latent_model.explained_variance_ratio_
print(f"Latent PCA: {latent_scores.shape[1]} components, "
      f"PC1+PC2 = {100*(latent_prop[0]+latent_prop[1]):.1f}% of variance")

### PCA

In [ ]:
# PC1 v PC2

pca_datasets = [(f"Landmarks", pca_sparse["x"],  pca_sparse["prop"]),
                (f"Dense correspondences", pca_dense["x"], pca_dense["prop"]),
                ("NSM latents", latent_scores, latent_prop)]

_ = pc_pair_grid(pca_datasets, 0, 1, "PC1 vs PC2 — sparse vs dense vs latents",
                 specimens=specimens, out_dir=OUT_DIR,
                 family_base_colors=family_base_colors,
                 outstem=f"{RUN}_pca_1v2_comparison", show_legend=False,
                 width=2100, height=650)

In [ ]:
# PC3 v PC4

_ = pc_pair_grid(pca_datasets, 2, 3, "PC3 vs PC4 — sparse vs dense vs latents",
                 specimens=specimens, out_dir=OUT_DIR,
                 family_base_colors=family_base_colors,
                 outstem=f"{RUN}_pca_3v4_comparison", show_legend=False,
                 width=2100, height=650)

### t-SNE

In [ ]:
tsne_sparse = tsne_scores(pca_sparse["x"])
tsne_dense  = tsne_scores(pca_dense["x"])
tsne_latent = tsne_scores(codes)

tsne_datasets = [(f"Landmarks", tsne_sparse, None),
                 (f"Dense correspondences", tsne_dense,  None),
                 ("NSM latents", tsne_latent, None)]

_ = pc_pair_grid(tsne_datasets, 0, 1, "t-SNE — sparse vs dense vs latents",
                 specimens=specimens, out_dir=OUT_DIR,
                 family_base_colors=family_base_colors,
                 axis_prefix="t-SNE ", outstem=f"{RUN}_tsne_comparison", show_legend=False)

### UMAP

In [ ]:
umap_sparse = umap_scores(pca_sparse["x"], n_neighbors=5, min_dist=2, spread=3.0, n_epochs=500, repulsion_strength=3.0)
umap_dense  = umap_scores(pca_dense["x"], n_neighbors=5, min_dist=2, spread=3.0, n_epochs=500, repulsion_strength=3.0)
umap_latent = umap_scores(codes, n_neighbors=5, min_dist=2, spread=3.0, n_epochs=500, repulsion_strength=3.0)

umap_datasets = [(f"Landmarks", umap_sparse, None),
                 (f"Dense correspondences", umap_dense,  None),
                 ("NSM latents", umap_latent, None)]

_ = pc_pair_grid(umap_datasets, 0, 1, "UMAP — sparse vs dense vs latents",
                 specimens=specimens, out_dir=OUT_DIR,
                 family_base_colors=family_base_colors,
                 axis_prefix="UMAP ", outstem=f"{RUN}_umap_comparison", show_legend=False)

In [ ]:
# Stitch PCA / t-SNE / UMAP comparison PNGs into one 3x3 publication panel

panel_files = [f"{RUN}_pca_1v2_comparison.png",
               f"{RUN}_pca_3v4_comparison.png",
               f"{RUN}_tsne_comparison.png",
               f"{RUN}_umap_comparison.png"]

try:
    font = ImageFont.truetype("Arial.ttf", 40)
except OSError:
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 40)   
    except OSError:
        font = ImageFont.load_default()

imgs = [Image.open(OUT_DIR / fname) for fname in panel_files]
widths, heights = zip(*(im.size for im in imgs))

panel_w = widths[0]
total_h = sum(heights)

COL_LABELS = ["Landmarks", "Dense correspondences", "NSM latents"]
COL_HEADER_H = 70

grid = Image.new("RGB", (panel_w, COL_HEADER_H + total_h), "white")
draw = ImageDraw.Draw(grid)

col_w = panel_w // len(COL_LABELS)
for i, label in enumerate(COL_LABELS):
    bbox = draw.textbbox((0, 0), label, font=font)
    tx = i * col_w + (col_w - (bbox[2] - bbox[0])) / 2
    draw.text((tx, COL_HEADER_H - 45), label, fill="black", font=font)

y = COL_HEADER_H
for im in imgs:
    grid.paste(im, (0, y))
    y += im.height

outpath = OUT_DIR / f"{RUN}_pca_tsne_umap_3x3_panel.png"
grid.save(outpath, dpi=(300, 300))
print(f"Wrote {outpath.resolve()}  ({grid.size[0]}x{grid.size[1]} px)")
grid

## Export PC/tSNE/UMAP coords for R / Excel---

In [ ]:
# One tidy CSV per (dataset, method) combination
export_scores("PC_sparse",   pca_sparse["x"], specimens, OUT_DIR)
export_scores("PC_dense",    pca_dense["x"],  specimens, OUT_DIR)
export_scores("PC_latent",   latent_scores,   specimens, OUT_DIR)
export_scores("tSNE_sparse", tsne_sparse, specimens, OUT_DIR, n_components=2)
export_scores("tSNE_dense",  tsne_dense,  specimens, OUT_DIR, n_components=2)
export_scores("tSNE_latent", tsne_latent, specimens, OUT_DIR, n_components=2)
export_scores("UMAP_sparse", umap_sparse, specimens, OUT_DIR, n_components=2)
export_scores("UMAP_dense",  umap_dense,  specimens, OUT_DIR, n_components=2)
export_scores("UMAP_latent", umap_latent, specimens, OUT_DIR, n_components=2)

print(f"Wrote to {OUT_DIR.resolve()}:")
for f in sorted(OUT_DIR.glob("*_points_for_stats.csv")):
    print("  ", f.name)